# Faster R-CNN Training Loop
This notebook isolates the Faster R-CNN PyTorch training process using the original data loaders and training loops.

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import sys
sys.path.append('../')
from src.data.dataset import COCODetectionDataset, collate_fn
from src.evaluation.metrics import plot_confusion_matrix_with_background

DATASET_ROOT = '/content/Military-vehicles-detection-1'
BATCH_SIZE = 4
NUM_CLASSES = 5
NUM_EPOCHS = 10


In [ ]:
# DataLoaders
train_dataset = COCODetectionDataset(
    images_dir=os.path.join(DATASET_ROOT, 'train'),
    json_path=os.path.join(DATASET_ROOT, 'train', 'remapped_annotations.json')
)
val_dataset = COCODetectionDataset(
    images_dir=os.path.join(DATASET_ROOT, 'valid'),
    json_path=os.path.join(DATASET_ROOT, 'valid', 'remapped_annotations.json')
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)


In [ ]:
def get_object_detection_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    return model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_object_detection_model(NUM_CLASSES)
model.to(device)


In [ ]:
# Original PyTorch Training Loop
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

USE_AMP = True
scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

history = {"train_loss": [], "val_loss": []}
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss_epoch = 0.0
    for batch_idx, (images, targets) in enumerate(train_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        with torch.cuda.amp.autocast():
            loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss_epoch += losses.item()
    
    avg_train_loss = train_loss_epoch / len(train_loader)
    history["train_loss"].append(avg_train_loss)
    
    print(f'Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}')
